In [1]:
import polars as pl
import numpy as np
from string import ascii_lowercase as letters
letters = letters+"_"

#word_dictionary_file_path stolen from:
# https://github.com/dwyl/english-words/blob/master/words_alpha.txt
word_dictionary_file_path = r"C:\Users\brett\Downloads\words_alpha.txt".replace("\\","/")

In [2]:
def word_to_matrix(word):
    numbers = list(map(int, range(1, len(letters)+2)))
    let_to_num = dict(zip(letters, numbers))
    num_to_let = dict(zip(numbers, letters))

    word = list(word)
    word = np.array(word)
    
    get_val = np.vectorize(lambda word: let_to_num.get(word, 'Unknown'))
    return np.array(get_val(word))

In [3]:
def word_finder(known, length=-1, contains="", does_not_contain="",lnixp={},pprint=True):
    with open(word_dictionary_file_path,'r') as file:
        words = file.read()
        words = words.split("\n")
    ids = range(len(words))
    df = pl.DataFrame({"index": ids,"words": words})
    word_lengths = [ len(word) for word in df["words"] ]
    df = df.with_columns(word_length=pl.Series(word_lengths))
    
    #length check
    if length != -1:
        df = df.filter(pl.col("word_length") != 0)
        df = df.filter(pl.col("word_length") == length)

    num_matricies = [word_to_matrix(word) for word in df['words']] #converts strings to matricies

    df = df.with_columns(num_matrix=pl.Series(num_matricies)) #puts the matricies in the df

    known_matrix = np.array(word_to_matrix(known)) #creates a code for our known word
    
    df = df.with_columns(truth_matrix=pl.Series([known_matrix==np.array(num_matrix) for num_matrix in df['num_matrix']])) #returns a matrix with where our matrix matches any other word's matrix and adds that column to the df
    df = df.with_columns(truth_sums=pl.Series([sum(truth_matrix) for truth_matrix in df['truth_matrix']])) #returns the sums of truths for all truth matricies
    df = df.filter(pl.col("truth_sums") == df["truth_sums"].max()) #filters the df to only contain the most matching matricies

    #filter for the does_not_contain variable
    if len(does_not_contain) != 0:
        dnc_matrix = np.array(word_to_matrix(does_not_contain))
        df = df.with_columns(dnc_overlap=pl.Series([set(dnc_matrix)&set(num_matrix) for num_matrix in df['num_matrix']]))
        df = df.with_columns(len_of_dnc_overlap=pl.Series([len(dnc_overlap) for dnc_overlap in df['dnc_overlap']]))
        df = df.filter(pl.col("len_of_dnc_overlap") == 0) #filters the df to only contain the most matching matricies

    
    #filter for the contains variable
    if len(contains) != 0:
        contains_matrix = np.array(word_to_matrix(contains))
        df = df.with_columns(contains_overlap=pl.Series([set(contains_matrix)&set(num_matrix) for num_matrix in df['num_matrix']]))
        df = df.with_columns(len_of_contains_overlap=pl.Series([len(contains_overlap) for contains_overlap in df['contains_overlap']]))
        df = df.filter(pl.col("len_of_contains_overlap") == len(contains)) #filters the df to only contain the most matching matricies

    #filter for the "lnixp" variable (letter_not_in_x_position)
    for key in lnixp.keys():
        if len(lnixp[key]) != 0:
            lnixp_matrix = np.array(word_to_matrix(lnixp[key]))
            
            df = df.with_columns(lnixp_overlap=pl.Series([set(lnixp_matrix)&set(num_matrix[key:key+1]) for num_matrix in df['num_matrix']]))
            df = df.with_columns(len_of_lnixp_overlap=pl.Series([len(lnixp_overlap) for lnixp_overlap in df['lnixp_overlap']]))
            df = df.filter(pl.col("len_of_lnixp_overlap") == 0) #filters the df to remove any words that contian letters in wrong positions

    #pretty printing vs. normal printing
    if pprint == False:
        for i,word in enumerate(df["words"]):
            print(f"{i:4}\t{word}")
    else:
        for i,word in enumerate(df["words"]):
            print(f"+-----+----------+")
            print(f"|{i:4} | {word}    |")
        print(f"+-----+----------+")

In [5]:
lnixp = {0:"",
         1:"",
         2:"",
         3:"",
         4:""}
word_finder("r__al",length=5,contains="",does_not_contain="stenowdyiv",lnixp=lnixp,pprint=False)

   0	rabal
   1	ramal
   2	rugal
   3	rumal
   4	rural
